# 콘크리트 압축강도 예측 — EDA와 모델 학습 (Concrete)

- 데이터: 콘크리트 배합·양생 기록 (1,030행 · 9열)
- 목표: 압축강도(MPa) 예측 — 회귀
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_Concrete.txt

- 이 데이터는 전처리 함정이 거의 없음(결측 0)
  → 대신 **도메인 파생변수(물-시멘트비)** 실습에 집중

## 1. 불러오기

- **.xls 엑셀 파일** → read_excel 사용 (read_csv 아님)
- 컬럼명이 매우 길고 단위가 붙어 있음 → 정리 필요

In [ ]:
import pandas as pd
import numpy as np
from autogluon.common.utils.resource_utils import ResourceManager
ResourceManager.get_cpu_count = staticmethod(lambda **kwargs: 16)

df = pd.read_excel("Concrete_Data.xls")
print(df.shape)          # (1030, 9)
df.head()

### 1-1. 컬럼명 정리

- 원본: 'Cement (component 1)(kg in a m^3 mixture)' 처럼 길다
- 짧고 다루기 쉬운 이름으로 교체 (순서는 위 head로 확인)

In [ ]:
df.columns = ["cement", "slag", "fly_ash", "water",
              "superplasticizer", "coarse_agg", "fine_agg",
              "age", "strength"]
print(df.columns.tolist())

## 2. 학습 전 확인

- 결측·인코딩은 AutoGluon이 자동 처리
- 이 데이터에서 사람이 볼 것: 0의 의미

### 2-1. 0은 결측이 아니라 실제 값

- slag·fly_ash·superplasticizer에 0이 많음 (36~55%)
- 이 0은 '미측정'이 아니라 **그 재료를 안 넣은 배합**
- 따라서 결측으로 바꾸면 안 됨 (Pump it Up과 반대)

In [ ]:
# 0 비율 확인 — 보조 재료라 0이 많은 게 정상
for c in ["slag", "fly_ash", "superplasticizer"]:
    print(f"{c}: 0비율 {(df[c]==0).mean():.1%}")

print("\n결측:", df.isna().sum().sum())   # 0이어야 정상

## 3. EDA

- ProfileReport로 분포·상관을 확인
- 특히 각 재료와 강도의 관계에 주목

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("concrete_eda.html")   # 브라우저에서 열기

### 3-1. target(강도) 분포

In [ ]:
import matplotlib.pyplot as plt
print(df["strength"].describe())

df["strength"].hist(bins=40, figsize=(8,3))
plt.title("compressive strength (MPa)")
plt.show()

### 3-2. 재료와 강도의 관계

- 시멘트가 많을수록 강도가 오르는가
- 양생기간(age)이 길수록 강도가 오르는가

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3))
df.plot.scatter(x="cement", y="strength", alpha=0.3, ax=ax[0])
df.plot.scatter(x="age", y="strength", alpha=0.3, ax=ax[1])
plt.show()

## 4. 도메인 파생변수 — 물-시멘트비

- 재료공학에서 **물-시멘트비(water/cement)**는 강도의 핵심 지표
- 물이 많을수록 강도가 떨어진다는 것이 정설
- 원본에는 없는 변수를 사람이 만들어 넣는다
  → AutoGluon이 못 하는 '도메인 지식' 영역

In [ ]:
df["water_cement_ratio"] = df["water"] / df["cement"]

# 물-시멘트비와 강도의 관계 확인
df.plot.scatter(x="water_cement_ratio", y="strength",
                alpha=0.3, figsize=(7,3),
                title="water/cement ratio vs strength")
plt.show()
# 비율이 높을수록 강도가 낮아지는 경향이 보이는가?

## 5. 학습 — 파생변수 효과 비교

- 파생변수 없이 vs 있이 학습해 성능을 비교
- 도메인 지식이 실제로 도움이 되는지 확인

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42)

# (A) 파생변수 포함
pred_A = TabularPredictor(
    label="strength",
    eval_metric="root_mean_squared_error",
).fit(train_data, presets="medium_quality", time_limit=300)

In [ ]:
# (B) 파생변수 제외 (water_cement_ratio 빼고 학습)
cols_B = [c for c in train_data.columns if c != "water_cement_ratio"]
pred_B = TabularPredictor(
    label="strength",
    eval_metric="root_mean_squared_error",
).fit(train_data[cols_B], presets="medium_quality", time_limit=300)

### 5-1. 두 모델 성능 비교

In [ ]:
perf_A = pred_A.evaluate(test_data)
perf_B = pred_B.evaluate(test_data[cols_B])
print("파생변수 포함(A):", perf_A)
print("파생변수 제외(B):", perf_B)
# A가 더 좋다면, 도메인 지식이 성능을 높인 것

## 6. 해석

In [ ]:
pred_A.leaderboard(test_data)

### 6-1. 변수 중요도

- water_cement_ratio가 상위에 오는가
- 원본 재료들과 비교해 얼마나 기여하는가

In [ ]:
pred_A.feature_importance(test_data)

## 정리

- .xls 엑셀 → read_excel · 긴 컬럼명 정리
- 0은 실제 값(안 넣은 재료) → 결측으로 바꾸지 않음
- 물-시멘트비 = 도메인 파생변수 → 사람이 만드는 영역
- 파생변수 포함/제외 성능 비교 → 도메인 지식의 가치 확인
- 학습·앙상블은 TabularPredictor가 자동

- Absenteeism과 비교: 그쪽은 전처리 함정(범주·분할)이 핵심,
  이쪽은 파생변수가 핵심 → 데이터마다 사람의 역할이 다르다